# Group-Aware Multilabel Preparation

Stratify approximately by complete label combination while keeping every patch from one source frame in one split.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
PROJECT_ROOT = Path.cwd().parents[1]
LABELS_FILE = PROJECT_ROOT / "data" / "raw" / "archive" / "classification" / "labels.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "multilabel"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_COLUMNS = ["Surface_Crack", "Delamination", "Pinhole"]

data = pd.read_csv(LABELS_FILE)
data["recognized_label_count"] = data[LABEL_COLUMNS].sum(axis=1)
data = data[(data["unclassified"] == 0) & (data["recognized_label_count"] > 0)].copy()
data["frame_group"] = data["original_file_name"].str.replace(r"_patch_\d+\.png$", "", regex=True)
data["label_combination"] = data[LABEL_COLUMNS].astype(str).agg("".join, axis=1)
print("Usable images:", len(data), "Source frames:", data["frame_group"].nunique())

Usable images: 2105 Source frames: 359


In [2]:
splitter = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
data["fold"] = -1

for fold_number, (_, fold_indices) in enumerate(
    splitter.split(data, y=data["label_combination"], groups=data["frame_group"])
):
    data.iloc[fold_indices, data.columns.get_loc("fold")] = fold_number

test_df = data[data["fold"] == 0].drop(columns="fold").copy()
val_df = data[data["fold"] == 1].drop(columns="fold").copy()
train_df = data[~data["fold"].isin([0, 1])].drop(columns="fold").copy()

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

In [3]:
for name, split_df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print()
    print(f"{name}: {len(split_df)} images, {split_df['frame_group'].nunique()} frames")
    print(split_df[LABEL_COLUMNS].sum())
    print(split_df["label_combination"].value_counts().sort_index())

train_groups = set(train_df["frame_group"])
val_groups = set(val_df["frame_group"])
test_groups = set(test_df["frame_group"])
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)
assert len(train_df) + len(val_df) + len(test_df) == len(data)
print()
print("Verified: no source-frame overlap.")


Train: 1503 images, 255 frames
Surface_Crack    1365
Delamination      146
Pinhole           358
dtype: int64
label_combination
001     121
010      17
100    1020
101     216
110     108
111      21
Name: count, dtype: int64

Validation: 302 images, 53 frames
Surface_Crack    274
Delamination      31
Pinhole           72
dtype: int64
label_combination
001     24
010      4
100    204
101     43
110     22
111      5
Name: count, dtype: int64

Test: 300 images, 51 frames
Surface_Crack    276
Delamination      26
Pinhole           73
dtype: int64
label_combination
001     24
100    206
101     44
110     21
111      5
Name: count, dtype: int64

Verified: no source-frame overlap.
